In [1]:
import pandas as pd
from bertopic import BERTopic

In [2]:
df = pd.read_csv('sem4data.csv')

C:\Users\janek\AppData\Local\Temp\ipykernel_41792\116698092.py:1: DtypeWarning: Columns (9,10,11,12,15,16,17,18,27,29,30,33,35,53,54,55,56,58,59,70,73) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('sem4data.csv')


In [3]:
tweets = 'tweets_historical'


In [4]:
df = df.dropna(subset=[tweets])
df = df[df[tweets].str.strip() != '']

df = df[~df[tweets].str.startswith("RT @")]

df = df[df[tweets].str.len() >= 20]

print(f'After filtering: {len(df[tweets])} posts')

After filtering: 2081342 posts


## Normalizing Text

In [5]:
import re

In [ ]:

def clean_tweet(text):
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove @mentions
    text = re.sub(r"@\w+", "", text)

    text = re.sub(r"#(\w+)", r"\1", text)

    text = re.sub(r"[^\w\s]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

df["tweet_clean"] = df[tweets].apply(clean_tweet)

df = df[df["tweet_clean"].str.len() >= 15]

print(f"After cleaning: {len(df)} posts")

After cleaning: 1779948 posts


In [7]:
df_sample = df.sample(n=50000, random_state=42)
docs = df_sample["tweet_clean"].dropna().tolist()


### Building Model

In [8]:
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

In [9]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    metric='cosine',
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size = 50,
    metric = 'euclidean',
    cluster_selection_method = 'eom',
    prediction_data = True
)

vectorizer_model = CountVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    min_df=2
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=True,
    top_n_words=10,
    verbose=True
)

In [10]:
topics, probs = topic_model.fit_transform(docs)

topic_info = topic_model.get_topic_info()
print(topic_info.head(10))

2026-03-31 08:00:52,179 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

2026-03-31 08:02:52,383 - BERTopic - Embedding - Completed ✓
2026-03-31 08:02:52,383 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-31 08:03:31,510 - BERTopic - Dimensionality - Completed ✓
2026-03-31 08:03:31,511 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-31 08:03:45,122 - BERTopic - Cluster - Completed ✓
2026-03-31 08:03:45,130 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-31 08:03:46,326 - BERTopic - Representation - Completed ✓


   Topic  Count                                               Name  \
0     -1  25718                          -1_trump_like_just_people   
1      0   3770                   0_morning_good_good morning_know   
2      1   1367                           1_israel_hamas_gaza_jews   
3      2   1359                         2_team_game_players_season   
4      3   1196                           3_taylor_like_just_woman   
5      4   1133                      4_tax_inflation_taxes_economy   
6      5   1018  5_healthcare_digitalhealth_patient_healthcare ...   
7      6    828                            6_god_jesus_church_cult   
8      7    637              7_thank following_following_hey_thank   
9      8    567                8_kamala_kamala harris_harris_trump   

                                      Representation  \
0  [trump, like, just, people, don, amp, biden, d...   
1  [morning, good, good morning, know, day, just,...   
2  [israel, hamas, gaza, jews, israeli, jewish, h...   
3  [t

In [11]:
df_sample = df_sample.rename(columns={"clean...state_simple": "state"})

In [12]:
print(df_sample.columns.tolist())


['created_at.users', 'protected', 'verified_type', 'public_metrics.followers_count', 'public_metrics.following_count', 'public_metrics.tweet_count', 'public_metrics.listed_count', 'public_metrics.like_count.users', 'public_metrics.media_count', 'entities.url.urls', 'entities.description.mentions', 'entities.description.hashtags', 'entities.description.urls', 'verified', 'created_at.tweets', 'entities.urls.subject_pool', 'entities.mentions.subject_pool', 'entities.annotations.subject_pool', 'entities.hashtags.subject_pool', 'sampling_tweet', 'lang.subject_pool', 'public_metrics.retweet_count.subject_pool', 'public_metrics.reply_count.subject_pool', 'public_metrics.like_count.tweets', 'public_metrics.quote_count.subject_pool', 'public_metrics.bookmark_count.subject_pool', 'public_metrics.impression_count.subject_pool', 'edit_history_tweet_ids.subject_pool', 'possibly_sensitive', 'attachments.media_keys.subject_pool', 'referenced_tweets.subject_pool', 'pool_type', 'sampling_date', 'entiti

### state level analysis

In [13]:
#topic distribution by state

df_sample['topic'] = topics

state_topics = df_sample.groupby(["state","topic"]).size().unstack(fill_value=0)
print(state_topics)


topic                  -1    0    1    2    3    4    5    6    7    8   ...  \
state                                                                    ...   
Alabama                 35   18    0    0    1    0    0    1    0    0  ...   
Arizona                519   79    6   32   20   19    2   83    0    5  ...   
Arkansas                62    5    0    0    4    1    0    0    0    0  ...   
California            3526  409  163   97  120  136   24   99    9   98  ...   
Colorado               587   73   32   20   17   43    3   53    0    8  ...   
Connecticut             77   13    0   19    3    0    0    1    0    2  ...   
Delaware                17    1    0    1    2    1    0    0    0    0  ...   
District of Columbia   361   57   74   20   12   24    0   10    0    7  ...   
Florida               3185  416   63  178   97  131   16   63  554   27  ...   
Georgia                544   74   40   22   40   18    3    8    0   18  ...   
Hawaii                 226   22    0    

In [14]:
#which states talk about which topics the most?
state_topics_pct = state_topics.div(state_topics.sum(axis=1), axis=0) * 100
print(state_topics_pct)

topic                       -1          0          1          2         3   \
state                                                                        
Alabama               53.846154  27.692308   0.000000   0.000000  1.538462   
Arizona               53.560372   8.152735   0.619195   3.302374  2.063983   
Arkansas              69.662921   5.617978   0.000000   0.000000  4.494382   
California            52.634722   6.105389   2.433199   1.447977  1.791312   
Colorado              53.315168   6.630336   2.906449   1.816530  1.544051   
Connecticut           52.739726   8.904110   0.000000  13.013699  2.054795   
Delaware              65.384615   3.846154   0.000000   3.846154  7.692308   
District of Columbia  48.586810   7.671602   9.959623   2.691790  1.615074   
Florida               53.189713   6.947228   1.052104   2.972612  1.619906   
Georgia               55.005056   7.482305   4.044489   2.224469  4.044489   
Hawaii                65.697674   6.395349   0.000000   1.453488

### Model Evaluation

In [15]:
#coherence score

from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

# Prepare data
cleaned_docs = [doc.split() for doc in docs]
dictionary = Dictionary(cleaned_docs)

topics_words = []
for topic_id in topic_model.get_topic_info()["Topic"]:
    if topic_id != -1:  # exclude noise
        words = [word for word, _ in topic_model.get_topic(topic_id)]
        # Keep only words that exist in the dictionary
        words = [word for word in words if word in dictionary.token2id]
        if len(words) > 0:  # only add if at least one word survived
            topics_words.append(words)

print(f"Number of valid topics: {len(topics_words)}")


# Calculate coherence
coherence_model = CoherenceModel(
    topics=topics_words,
    texts=cleaned_docs,
    dictionary=dictionary,
    coherence="c_v"  # c_v is the most reliable metric
)
coherence_score = coherence_model.get_coherence()
print(f"Coherence score (c_v): {coherence_score:.4f}")

Number of valid topics: 81
Coherence score (c_v): 0.4609


In [16]:
#Topic Diversity

def topic_diversity(topics_words, top_n=10):
    unique_words = set()
    total_words = []
    for topic in topics_words:
        top_words = topic[:top_n]
        unique_words.update(top_words)
        total_words.extend(top_words)
    return len(unique_words) / len(total_words)

diversity_score = topic_diversity(topics_words)
print(f"Topic diversity: {diversity_score:.4f}")


Topic diversity: 0.8972


In [17]:
#number of outliers
n_outliers = sum(1 for t in topics if t == -1)
outlier_pct = (n_outliers / len(topics)) * 100
print(f"Outliers: {n_outliers} ({outlier_pct:.1f}%)")

Outliers: 25718 (51.4%)


## Visualisations

In [25]:

topic_map = topic_info.set_index("Topic")["Name"].to_dict()

# Add topic_name column to df_sample
df_sample["topic_name"] = df_sample["topic"].map(topic_map)


In [26]:
custom_names = {
    0  : "General Greetings / Spam",
    1  : "Middle East / Israel-Gaza",
    2  : "Sports",
    3  : "Taylor Swift / Pop Culture",
    4  : "Economic Policy",
    5  : "Healthcare / Digital Health",
    6  : "Religion / Faith",
    7  : "Spam / Engagement Bait",
    8  : "2024 Election / Candidates",
    -1 : "Noise"
}

# Filter out non-political topics as well as noise
non_political = [0, 2, 3, 7]  # greetings, sports, taylor swift, spam

df_sample["topic_name_custom"] = df_sample["topic"].map(custom_names)
df_viz = df_sample[
    (~df_sample["topic"].isin(non_political)) & 
    (df_sample["topic"] != -1)
].copy()

print(f"\nTopic counts:\n{df_sample['topic_name_custom'].value_counts()}")
print(f"\nPolitical topics only: {len(df_viz)} tweets")


Topic counts:
topic_name_custom
Noise                          25718
General Greetings / Spam        3770
Middle East / Israel-Gaza       1367
Sports                          1359
Taylor Swift / Pop Culture      1196
Economic Policy                 1133
Healthcare / Digital Health     1018
Religion / Faith                 828
Spam / Engagement Bait           637
2024 Election / Candidates       567
Name: count, dtype: int64

Political topics only: 17320 tweets


In [27]:
import plotly.express as px

In [28]:
topic_counts = df_viz["topic_name_custom"].value_counts().reset_index()
topic_counts.columns = ["topic", "count"]

fig = px.bar(
    topic_counts,
    x="topic",
    y="count",
    title="Tweet volume per policy issue",
    labels={"topic": "Policy issue", "count": "Number of tweets"},
    color="topic"
)
fig.update_layout(xaxis_tickangle=45, showlegend=False)
fig.show()


In [29]:
state_name_to_code = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "Florida": "FL", "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID",
    "Illinois": "IL", "Indiana": "IN", "Iowa": "IA", "Kansas": "KS",
    "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
    "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV",
    "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY",
    "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK",
    "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT",
    "Vermont": "VT", "Virginia": "VA", "Washington": "WA", "West Virginia": "WV",
    "Wisconsin": "WI", "Wyoming": "WY", "District of Columbia": "DC"
}

# Apply to your dataframe
df_sample["state"] = df_sample["state"].map(state_name_to_code)



In [30]:
df_viz = df_viz[~df_viz["state"].isin(["USA", "US", "United States", None])]
# Calculate what % of each topic's tweets come from each state
topic_state_counts = (
    df_viz.groupby(["topic_name_custom", "state"])
    .size()
    .reset_index(name="count")
)

# Add % within each topic
topic_state_counts["pct"] = (
    topic_state_counts.groupby("topic_name_custom")["count"]
    .transform(lambda x: x / x.sum() * 100)
    .round(1)
)

fig = px.treemap(
    topic_state_counts,
    path=["topic_name_custom", "state"],
    values="count",
    custom_data=["pct", "state"],
    title="States grouped by policy issue",
    color="topic_name_custom",
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig.update_traces(
    texttemplate="%{customdata[1]}<br>%{customdata[0]}%",
    textposition="middle center"
)

fig.update_layout(height=700)
fig.show()

In [31]:


# Calculate % of each topic per state
state_topic = (
    df_viz.groupby(["state", "topic_name_custom"])
    .size()
    .reset_index(name="count")
)

state_totals = df_viz.groupby("state").size().reset_index(name="total")
state_topic = state_topic.merge(state_totals, on="state")
state_topic["pct"] = (state_topic["count"] / state_topic["total"] * 100).round(1)


fig = px.bar(
    state_topic,
    x="state",
    y="pct",
    color="topic_name_custom",
    title="Policy issue composition by state",
    labels={
        "pct": "% of tweets",
        "state": "State",
        "topic_name_custom": "Policy issue"
    },
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig.update_layout(
    barmode="stack",
    xaxis=dict(tickangle=45, tickfont=dict(size=9)),
    yaxis=dict(title="% of tweets", gridcolor="lightgrey"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    height=600,
    legend=dict(
        title="Policy issue",
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.01
    ),
    margin=dict(t=80, b=120, l=60, r=200)
)

fig.show()